In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.train_transactions AS 
SELECT *
FROM read_files(
    '/Volumes/workspace/default/train_transactions/train_transaction.csv', 
    format => 'csv', 
    header => true
)


In [0]:
%sql
DESCRIBE workspace.default.train_transactions;

In [0]:
%sql
Select * 
FROM workspace.default.train_transactions
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.train_transactions_analysis AS
SELECT 
    TransactionID, 
    isFraud, 
    TransactionDT, 
    TransactionAmt, 
    ProductCD, 
    card1, 
    card4, 
    card6,
    P_emaildomain, 
    R_emaildomain, 
    date_add('2017-12-01', cast(floor(TransactionDT/86400) as int)) as TransactionDate,
    CAST(FLOOR(TransactionDT / 3600) % 24 AS INT) as TransactionHour, 
    COUNT(*) OVER (
        Partition by card1
        Order By TransactionDT
        RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
    ) as TransactionsLastHour
FROM workspace.default.train_transactions

In [0]:
%sql
SELECT * FROM train_transactions_analysis LIMIT 20

In [0]:
%sql
/*Check for duplicate transaction ids*/
Select 
    TransactionID,
    Count(*) as duplicate_count
from train_transactions_analysis
group by TransactionID
Having Count(*) > 1



In [0]:
%sql
/* Data Quality Checks */
SELECT
    'Duplicate Transaction IDs' as check_name, 
    COUNT(*) - COUNT(DISTINCT TransactionID) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN COUNT(*) - COUNT(DISTINCT TransactionID) = 0 THEN 'PASS' ELSE 'FAIl' END as status
from workspace.default.train_transactions_analysis

UNION ALL

SELECT 
    'Missing or non-positive amounts' as check_name, 
    count_if(TransactionAmt is null or TransactionAmt <= 0) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN count_if(TransactionAmt is null or TransactionAmt <= 0) = 0 THEN 'PASS' ELSE 'FAIL' END as status
FROM workspace.default.train_transactions_analysis

UNION ALL

SELECT 
    'Missing emails' as check_name, 
    count_if(P_emaildomain is null) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN count_if(P_emaildomain is null) = 0 THEN 'PASS' ELSE 'WARN' END as status
FROM workspace.default.train_transactions_analysis

UNION ALL 

SELECT 
    'Bad Labels' as check_name, 
    count_if(isFraud not in (0,1) or isFraud is null) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN count_if(isFraud not in (0,1) or isFraud is null) = 0 THEN 'PASS' ELSE 'FAIL' END as status
FROM workspace.default.train_transactions_analysis

UNION ALL

SELECT
  'Row count matches source' AS check_name,
  ABS((SELECT COUNT(*) FROM workspace.default.train_transactions)
    - (SELECT COUNT(*) FROM workspace.default.train_transactions_analysis)) AS failing_rows,
  ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
  CASE WHEN (SELECT COUNT(*) FROM workspace.default.train_transactions)
          = (SELECT COUNT(*) FROM workspace.default.train_transactions_analysis)
       THEN 'PASS' ELSE 'FAIL' END AS status
From workspace.default.train_transactions_analysis

In [0]:
%sql
/*Create view of data quality check*/

CREATE OR REPLACE VIEW workspace.default.train_transactions_analysis_data_checks AS
SELECT
    'Duplicate Transaction IDs' as check_name, 
    COUNT(*) - COUNT(DISTINCT TransactionID) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN COUNT(*) - COUNT(DISTINCT TransactionID) = 0 THEN 'PASS' ELSE 'FAIL' END as status
from workspace.default.train_transactions_analysis

UNION ALL

SELECT 
    'Missing or non-positive amounts' as check_name, 
    count_if(TransactionAmt is null or TransactionAmt <= 0) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN count_if(TransactionAmt is null or TransactionAmt <= 0) = 0 THEN 'PASS' ELSE 'FAIL' END as status
FROM workspace.default.train_transactions_analysis

UNION ALL

SELECT 
    'Missing emails' as check_name, 
    count_if(P_emaildomain is null) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN count_if(P_emaildomain is null) = 0 THEN 'PASS' ELSE 'WARN' END as status
FROM workspace.default.train_transactions_analysis

UNION ALL 

SELECT 
    'Bad Labels' as check_name, 
    count_if(isFraud not in (0,1) or isFraud is null) as failing_rows,
    ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
    CASE WHEN count_if(isFraud not in (0,1) or isFraud is null) = 0 THEN 'PASS' ELSE 'FAIL' END as status
FROM workspace.default.train_transactions_analysis

UNION ALL

SELECT
  'Row count matches source' AS check_name,
  ABS((SELECT COUNT(*) FROM workspace.default.train_transactions)
    - (SELECT COUNT(*) FROM workspace.default.train_transactions_analysis)) AS failing_rows,
  ROUND(failing_rows/ COUNT(*) * 100, 2) as pct_failing,
  CASE WHEN (SELECT COUNT(*) FROM workspace.default.train_transactions)
          = (SELECT COUNT(*) FROM workspace.default.train_transactions_analysis)
       THEN 'PASS' ELSE 'FAIL' END AS status
From workspace.default.train_transactions_analysis


In [0]:
%sql
 DESCRIBE workspace.default.train_transactions_analysis_data_checks

In [0]:
%sql
Select * from workspace.default.train_transactions_analysis_data_checks

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.train_transactions_headline_numbers AS
SELECT
    COUNT(*) as total_txns, 
    SUM(isFraud) as fraud_txns, 
    ROUND(AVG(isFraud) * 100, 2) as fraud_rate_pct, 
    ROUND(SUM(TransactionAmt), 2) as total_amt, 
    ROUND(SUM(CASE WHEN isFraud = 1 THEN TransactionAmt END), 2) as fraud_amt, 
    ROUND((fraud_amt / total_amt) * 100, 2) as fraud_amt_pct, 
    ROUND(AVG(CASE WHEN isFraud = 1 THEN TransactionAmt END), 2) as avg_fraud_ticket,
    ROUND(AVG(CASE WHEN isFraud = 0 THEN TransactionAmt END), 2) as avg_good_ticket
FROM workspace.default.train_transactions_analysis


In [0]:
%sql
Select * from workspace.default.train_transactions_headline_numbers

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.default.train_transactions_analysis_weekly_trends AS

WITH weekly as (
    SELECT 
    CAST(DATE_TRUNC('week', TransactionDate) AS DATE) as week,
    COUNT(*) as total_txns, 
    SUM(isFraud) as fraud_txns, 
    ROUND(AVG(isFraud) * 100, 2) as fraud_rate_pct, 
    ROUND(SUM(CASE WHEN isFraud = 1 THEN TransactionAmt END), 2) as fraud_amt
    FROM workspace.default.train_transactions_analysis
    GROUP BY week 
)
SELECT 
week,
total_txns,
fraud_txns, 
fraud_rate_pct, 
fraud_amt,
ROUND(AVG(fraud_amt) OVER (
    Order by week
    ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
), 2) as avg_fraud_amt_4wk
FROM weekly





In [0]:
%sql
select * from workspace.default.train_transactions_analysis_weekly_trends

In [0]:
%sql 
CREATE OR REPLACE VIEW workspace.default.train_transactions_analysis_fraud_concentration AS
WITH fraud_concentration AS(
    SELECT
        ProductCD,
        card4,
        card6,
        COUNT(*) as total_txns, 
        SUM(isFraud) as fraud_txns, 
        ROUND(AVG(isFraud) * 100, 2) as fraud_rate_pct, 
        ROUND(SUM(CASE WHEN isFraud = 1 THEN TransactionAmt END), 2) as fraud_amt
    FROM workspace.default.train_transactions_analysis
    GROUP BY ProductCD, card4, card6
    HAVING COUNT(*) >= 1000
)
SELECT
    ProductCD, 
    card4, 
    card6,
    total_txns, 
    fraud_txns, 
    fraud_rate_pct, 
    fraud_amt, 
    ROUND(fraud_amt / SUM(fraud_amt) OVER(), 2) as share_fraud_amt, 
    ROUND((fraud_txns / total_txns) / (SUM(fraud_txns) OVER() / SUM(total_txns) OVER()), 2) as lift
FROM fraud_concentration





In [0]:
%sql
SELECT * FROM workspace.default.train_transactions_analysis_fraud_concentration
ORDER BY lift DESC

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.train_transactions_analysis_rule_tests AS 

WITH totals AS (
    SELECT SUM(CASE WHEN isFraud = 1 THEN TransactionAmt END) as total_fraud_amt, 
    COUNT_IF(isFraud = 1) as total_fraud_txns, 
    COUNT_IF(isFraud = 0) AS total_good_txns
    FROM workspace.default.train_transactions_analysis
),
velocity AS (
    SELECT
        concat('VELOCITY >= ', n, ' txns/hr') AS rule_name,
        'VELOCITY' AS rule_type,
        n as threshold, 
        COUNT(*) AS total_txns,
        COUNT_IF(t.TransactionsLastHour >= n) AS flagged, 
        COUNT_IF(t.TransactionsLastHour >= n AND t.isFraud = 1) AS fraud_caught,
        COUNT_IF(t.TransactionsLastHour >= n AND t.isFraud = 0) AS good_flagged,
        SUM(CASE WHEN t.TransactionsLastHour >= n AND t.isFraud = 1 THEN t.TransactionAmt ELSE 0 END) AS fraud_amt_caught
    FROM workspace.default.train_transactions_analysis as t
    CROSS JOIN (SELECT explode(sequence(1, 5)) AS n) v 
    GROUP BY n
), 
amount AS (
    SELECT 
        CONCAT('Amount > $', x) AS rule_name, 
        'AMOUNT' AS rule_type, 
        x as threshold, 
        COUNT(*) as total_txns, 
        COUNT_IF(t.TransactionAmt > x) AS flagged, 
        COUNT_IF(t.TransactionAmt > x AND t.isFraud = 1) AS fraud_caught, 
        COUNT_IF(t.TransactionAmt > x and t.isFraud = 0) AS good_flagged, 
        SUM(CASE WHEN t.TransactionAmt > x AND t.isFraud = 1 THEN t.TransactionAmt ELSE 0 END) AS fraud_amt_caught
    FROM workspace.default.train_transactions_analysis as t
    CROSS JOIN (VALUES (100), (200), (500), (1000), (2000)) as a(x)
    GROUP BY x
), 
all_rules AS (
    SELECT * FROM velocity
    UNION ALL
    SELECT * FROM amount
)
SELECT
    rule_name,
    rule_type, 
    threshold, 
    flagged, 
    ROUND(flagged / total_txns  * 100, 2) as pct_flagged, 
    fraud_caught, 
    ROUND(fraud_caught / total_fraud_txns * 100, 2) as recall_pct, 
    ROUND(fraud_amt_caught / total_fraud_amt * 100, 2) as pct_fraud_amt_caught, 
    good_flagged, 
    ROUND(good_flagged / total_good_txns * 100, 2) as pct_good_declined, 
    ROUND(fraud_caught / NULLIF(flagged, 0) * 100, 2) as precision_pct, 
    ROUND(good_flagged / NULLIF(fraud_caught, 0), 1) as good_per_fraud_caught
FROM all_rules
CROSS JOIN totals




In [0]:
%sql
SELECT * FROM workspace.default.train_transactions_analysis_rule_tests
ORDER BY rule_type, threshold